# Caption inspection — DroneWaste + AerialWaste

Review each image with its agent-generated caption and GT labels. Reads the **corrected** `full.jsonl` (9,297 captions; keys/labels repaired).

**How to use**: run all cells → use the slider viewer (one image at a time) or `show_grid`. Narrow with the filter cell (dataset, positives/negatives, a specific label, or **`ONLY="flagged"`** to review the 17 GT-vs-caption disagreements first). Flag bad ones with `flag(i,"reason")` then `save_flags()`.

In [ ]:
import json, random, textwrap
from collections import Counter
from pathlib import Path
import matplotlib.pyplot as plt
from PIL import Image

CAP = Path("/home/ids/diecidue/data/captions/full.jsonl")
FLAG = Path("/home/ids/diecidue/data/captions/flagged_captions.jsonl")
records = [json.loads(l) for l in CAP.read_text().splitlines() if l.strip()]
flagged_keys = {(r["dataset"], str(r["image_id"]))
                for r in (json.loads(l) for l in FLAG.read_text().splitlines() if l.strip())} if FLAG.exists() else set()
print(f"loaded {len(records)} captions; {len(flagged_keys)} pre-flagged (GT-vs-caption disagreements)")
print("by dataset :", dict(Counter(r["dataset"] for r in records)))
print("positives  :", sum(1 for r in records if r["gt_categories"]), " negatives:", sum(1 for r in records if not r["gt_categories"]))

In [ ]:
# ---- choose what to inspect (edit, then re-run this cell + the viewer) ----
DATASET = None        # "aerialwaste_m2" | "dronewaste_paper10" | None
ONLY    = "flagged"   # "all" | "positives" | "negatives" | "flagged"
LABEL   = None         # e.g. "Pallets"
SHUFFLE = False

def _keep(r):
    k=(r["dataset"], str(r["image_id"]))
    if DATASET and r["dataset"]!=DATASET: return False
    if ONLY=="positives" and not r["gt_categories"]: return False
    if ONLY=="negatives" and r["gt_categories"]: return False
    if ONLY=="flagged" and k not in flagged_keys: return False
    if LABEL and LABEL not in r["gt_categories"]: return False
    return True

view=[r for r in records if _keep(r)]
if SHUFFLE: random.seed(0); random.shuffle(view)
print(f"{len(view)} images in current view (ONLY={ONLY!r})")

In [ ]:
# ---- one-at-a-time viewer ----
from ipywidgets import interact, IntSlider
def render(i):
    if not view: print("empty view"); return
    r=view[i]
    try: img=Image.open(r["image_path"]).convert("RGB")
    except Exception as e: print("cannot open", r["image_path"], e); return
    fig,ax=plt.subplots(figsize=(7,7)); ax.imshow(img); ax.axis("off")
    ax.set_title(f"[{i}/{len(view)-1}]  {r['dataset']} / {r['image_id']}", fontsize=10); plt.show()
    print("GT labels:", r["gt_categories"] if r["gt_categories"] else "(none — negative)")
    print("-"*95); print(textwrap.fill(r["caption"], 100))
interact(render, i=IntSlider(0,0,max(0,len(view)-1),1,description="idx"));

In [ ]:
# ---- grid scan ----
def show_grid(start=0,n=12,cols=3,max_side=320):
    sub=view[start:start+n]; rows=(len(sub)+cols-1)//cols
    fig,axes=plt.subplots(rows,cols,figsize=(cols*4,rows*4))
    axes=axes.ravel() if hasattr(axes,"ravel") else [axes]
    for ax in axes: ax.axis("off")
    for k,r in enumerate(sub):
        img=Image.open(r["image_path"]).convert("RGB"); img.thumbnail((max_side,max_side)); axes[k].imshow(img)
        lab=", ".join(r["gt_categories"]) if r["gt_categories"] else "none"
        axes[k].set_title(f"[{start+k}] {r['image_id']}\n{lab}", fontsize=8)
    plt.tight_layout(); plt.show()
show_grid(0, min(12,len(view)))

In [ ]:
# ---- three ways to handle a caption you think is wrong ----
import shutil

def _persist():
    with open(CAP, "w") as f:
        for r in records:
            f.write(json.dumps(r, ensure_ascii=False) + "\n")

# 1) EDIT — you hand-write the corrected description; Labels line re-appended from GT.
def edit(i, new_text):
    bak = CAP.with_suffix(".jsonl.bak")
    if not bak.exists():
        shutil.copy(CAP, bak); print(f"backup -> {bak}")
    r = view[i]
    gt = sorted(r["gt_categories"]); labels = ", ".join(gt) if gt else "none"
    r["caption"] = new_text.strip() + "\n  Labels: " + labels
    _persist()
    print(f"edited [{i}] {r['dataset']}/{r['image_id']} -> saved")
    print(textwrap.fill(r["caption"], 100))

# 2) SUGGEST — queue a hint; an agent re-reads the image and rewrites the caption
#    coherently using your note (run the agent pass afterwards, then re-load).
QUEUE = Path("/home/ids/diecidue/data/captions/recaption_queue.jsonl")
def suggest(i, note):
    r = view[i]
    rec = {k: r[k] for k in ("dataset","image_id","image_path","gt_categories")}
    rec["old_caption"] = r["caption"]; rec["suggestion"] = note
    with open(QUEUE, "a") as f:
        f.write(json.dumps(rec, ensure_ascii=False) + "\n")
    print(f"queued [{i}] {r['dataset']}/{r['image_id']} for re-annotation: {note!r}")

def show_queue():
    if not QUEUE.exists(): print("queue empty"); return
    for ln in QUEUE.read_text().splitlines():
        q = json.loads(ln); print(f"  {q['dataset']}/{q['image_id']}: {q['suggestion']}")

# 3) FLAG — just record for later, no change.
flagged = []
def flag(i, reason=""):
    r = view[i]
    flagged.append({k: r[k] for k in ("dataset","image_id","image_path","gt_categories","caption")} | {"reason": reason})
    print(f"flagged [{i}] {r['dataset']}/{r['image_id']}: {reason}")
def save_flags(path="/home/ids/diecidue/data/captions/flagged_manual.jsonl"):
    with open(path, "w") as f:
        for x in flagged: f.write(json.dumps(x, ensure_ascii=False) + "\n")
    print(f"saved {len(flagged)} -> {path}")

# examples:
#   edit(0, "A grey corrugated asbestos-cement roof fills the frame; no loose waste present.")
#   suggest(0, "this is asbestos roofing, not scrap — describe the roof, not a debris pile")

In [ ]:
# ---- interactive review panel: image + text box + buttons ----
# Run this cell, then use the box + buttons below (no need to type function calls).
import ipywidgets as W
from IPython.display import display, clear_output

_state = {"i": 0}
out_img = W.Output()
status  = W.Output()
note = W.Textarea(
    placeholder="Type a SUGGESTION (for re-annotation by the agent) "
                "or a full NEW DESCRIPTION (for a direct edit)...",
    layout=W.Layout(width="95%", height="90px"))

def _draw():
    with out_img:
        clear_output(wait=True)
        if not view:
            print("empty view — relax the filter cell"); return
        r = view[_state["i"]]
        try:
            img = Image.open(r["image_path"]).convert("RGB")
        except Exception as e:
            print("cannot open", r["image_path"], e); return
        fig, ax = plt.subplots(figsize=(6, 6)); ax.imshow(img); ax.axis("off")
        ax.set_title(f"[{_state['i']}/{len(view)-1}]  {r['dataset']} / {r['image_id']}", fontsize=10)
        plt.show(); plt.close(fig)
        print("GT labels:", r["gt_categories"] if r["gt_categories"] else "(none — negative)")
        print("-" * 95); print(textwrap.fill(r["caption"], 100))

def _go(delta):
    _state["i"] = max(0, min(_state["i"] + delta, len(view) - 1)); _draw()

b_prev = W.Button(description="◀ Prev")
b_next = W.Button(description="Next ▶")
jump   = W.IntText(value=0, layout=W.Layout(width="90px"))
b_jump = W.Button(description="Go to idx")
b_sugg = W.Button(description="Queue suggest", button_style="warning",
                  tooltip="queue this note for agent re-annotation")
b_edit = W.Button(description="Apply edit", button_style="primary",
                  tooltip="overwrite the description with the box text now")
b_flag = W.Button(description="Flag", tooltip="record for later, no change")

b_prev.on_click(lambda b: _go(-1))
b_next.on_click(lambda b: _go(1))
b_jump.on_click(lambda b: (_state.update(i=max(0, min(jump.value, len(view) - 1))), _draw()))

def _suggest(b):
    with status:
        clear_output(wait=True)
        if not note.value.strip(): print("type a suggestion in the box first"); return
        suggest(_state["i"], note.value.strip()); note.value = ""
def _edit(b):
    with status:
        clear_output(wait=True)
        if not note.value.strip(): print("type the new description in the box first"); return
        edit(_state["i"], note.value.strip()); note.value = ""; _draw()
def _flag(b):
    with status:
        clear_output(wait=True); flag(_state["i"], note.value.strip())
b_sugg.on_click(_suggest); b_edit.on_click(_edit); b_flag.on_click(_flag)

display(W.VBox([
    W.HBox([b_prev, b_next, jump, b_jump]),
    out_img,
    note,
    W.HBox([b_sugg, b_edit, b_flag]),
    status,
]))
_draw()